# MoneyPrinterTurbo GitHub Issues queue — veilige Colab-versie

Deze notebook start alleen de GitHub Issues-wachtrij. Hij gebruikt **niet** poort 8080 en voert geen `fuser -k` of brede `pkill`-opdrachten uit. Daardoor blijft de Colab-runtimeverbinding intact.

Volg stap 1, 2 en 3 in deze volgorde. Stap 4 is alleen voor diagnose.


## 1. Installeer de actuele fork

Deze cel zet een bestaande checkout altijd om naar `Equilibriumpress/MoneyPrinterTurbo`, haalt de actuele `main` op en controleert of de queue-worker aanwezig is.


In [ ]:

import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/Equilibriumpress/MoneyPrinterTurbo.git"
REPO_BRANCH = "main"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} bestaat, maar is geen Git-repository. "
        "Kies Runtime → Disconnect and delete runtime en probeer opnieuw."
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

worker_path = REPO_DIR / "scripts" / "github_issue_worker.py"
if not worker_path.is_file():
    raise RuntimeError(f"Queue-worker ontbreekt: {worker_path}")

os.chdir(REPO_DIR)

subprocess.run(
    ["python", "-m", "pip", "install", "-q", "uv", "pyngrok"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(
    ["uv", "sync", "--frozen", "--python", "3.11"],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
origin = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
    text=True,
).strip()

print(f"Repository: {origin}")
print(f"Commit: {commit}")
print(f"Worker: {worker_path}")
print("Installatie gereed.")


## 2. Valideer ngrok en GitHub

De tokens worden verborgen ingevoerd. De cel controleert het GitHub-account, de repositorytoegang en het aantal opdrachten met label `video-job`.


In [ ]:

import json
from getpass import getpass
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

from pyngrok import ngrok

# Sluit alleen ngrok-tunnels uit deze Python-runtime.
try:
    ngrok.kill()
except Exception:
    pass

ngrok_token = getpass("Enter the ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("Een ngrok-token is verplicht")
ngrok.set_auth_token(ngrok_token)
del ngrok_token

github_token = getpass("Enter the fine-grained GitHub token: ").strip()
if not github_token:
    raise ValueError("Een GitHub-token is verplicht")

def github_get(path):
    request = Request(
        f"https://api.github.com{path}",
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {github_token}",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "MoneyPrinterTurbo-Colab-Queue",
        },
    )
    try:
        with urlopen(request, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub-tokencontrole mislukt met HTTP {exc.code}: {detail}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"GitHub is niet bereikbaar: {exc}") from exc

profile = github_get("/user")
repository = github_get("/repos/Equilibriumpress/MoneyPrinterTurbo")
queued = github_get(
    "/repos/Equilibriumpress/MoneyPrinterTurbo/issues"
    "?state=open&labels=video-job&per_page=30"
)

queued_count = len([item for item in queued if "pull_request" not in item])

print(f"GitHub account: {profile.get('login')}")
print(f"Repository access: {repository.get('full_name')}")
print(f"Queued video jobs: {queued_count}")
print("ngrok en GitHub-authenticatie zijn geldig.")


## 3. Start de API en de GitHub Issues-wachtrij

Deze cel kiest automatisch een vrije poort tussen 18080 en 18180. Hij beëindigt alleen processen die door een eerdere uitvoering van deze notebook zijn gestart.


In [ ]:

import os
import socket
import subprocess
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

API_LOG_PATH = Path("/content/moneyprinterturbo-api-safe.log")
WORKER_LOG_PATH = Path("/content/moneyprinterturbo-github-worker-safe.log")

def stop_known_process(name):
    process = globals().get(name)
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)

stop_known_process("github_worker_proc")
stop_known_process("api_proc")

for log_name in ("github_worker_log", "api_log"):
    handle = globals().get(log_name)
    if handle is not None and not handle.closed:
        handle.close()

previous_api_tunnel = globals().get("api_tunnel")
if previous_api_tunnel is not None:
    try:
        ngrok.disconnect(previous_api_tunnel.public_url)
    except Exception:
        pass

def find_free_port(start=18080, end=18180):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("127.0.0.1", port))
            except OSError:
                continue
            return port
    raise RuntimeError(f"Geen vrije poort gevonden tussen {start} en {end}")

API_PORT = find_free_port()
print(f"Veilige API-poort: {API_PORT}")

api_env = os.environ.copy()
api_env["PYTHONUNBUFFERED"] = "1"

api_log = API_LOG_PATH.open("w", encoding="utf-8")
api_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "uvicorn",
        "app.asgi:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(API_PORT),
        "--log-level",
        "warning",
    ],
    cwd=REPO_DIR,
    env=api_env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
    text=True,
)

deadline = time.time() + 300
next_log_report = time.time() + 30
api_ready = False

while time.time() < deadline:
    try:
        with urlopen(
            f"http://127.0.0.1:{API_PORT}/openapi.json",
            timeout=2,
        ) as response:
            api_ready = response.status == 200
    except (URLError, TimeoutError):
        pass

    if api_ready or api_proc.poll() is not None:
        break

    if time.time() >= next_log_report:
        api_log.flush()
        recent = API_LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )[-1600:]
        print("API start nog. Laatste log:")
        print(recent or "(nog geen loguitvoer)")
        next_log_report = time.time() + 30

    time.sleep(2)

if not api_ready:
    api_log.flush()
    recent_log = API_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
    raise RuntimeError(
        "MoneyPrinterTurbo API kon niet starten.\n"
        f"Process return code: {api_proc.poll()}\n"
        f"Recente log:\n{recent_log}"
    )

api_tunnel = ngrok.connect(
    addr=f"http://127.0.0.1:{API_PORT}",
    proto="http",
    bind_tls=True,
)

worker_env = os.environ.copy()
worker_env["PYTHONUNBUFFERED"] = "1"
worker_env["MPT_GITHUB_TOKEN"] = github_token
worker_env["MPT_GITHUB_REPOSITORY"] = "Equilibriumpress/MoneyPrinterTurbo"
worker_env["MPT_API_BASE"] = f"http://127.0.0.1:{API_PORT}/api/v1"
worker_env["MPT_PUBLIC_BASE"] = api_tunnel.public_url
worker_env["MPT_MAX_VIDEO_COUNT"] = "1"

github_worker_log = WORKER_LOG_PATH.open("w", encoding="utf-8")
github_worker_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "python",
        "-u",
        "scripts/github_issue_worker.py",
        "--poll-seconds",
        "20",
    ],
    cwd=REPO_DIR,
    env=worker_env,
    stdout=github_worker_log,
    stderr=subprocess.STDOUT,
    text=True,
)

worker_ready = False
deadline = time.time() + 45

while time.time() < deadline:
    if github_worker_proc.poll() is not None:
        break

    github_worker_log.flush()
    worker_text = WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )
    if "watches Equilibriumpress/MoneyPrinterTurbo" in worker_text:
        worker_ready = True
        break

    time.sleep(2)

if not worker_ready:
    github_worker_log.flush()
    recent_log = WORKER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )[-12000:]
    raise RuntimeError(
        "GitHub-worker kon niet starten.\n"
        f"Process return code: {github_worker_proc.poll()}\n"
        f"Recente log:\n{recent_log}"
    )

print("GitHub Issues queue is running.")
print(f"Lokale API: http://127.0.0.1:{API_PORT}")
print(f"Tijdelijke resultaat-URL: {api_tunnel.public_url}")
print(f"API-documentatie: {api_tunnel.public_url}/docs")
print("Bestaande issues met label video-job worden automatisch verwerkt.")


## 4. Diagnose

Voer deze cel uit wanneer een Issue niet naar `video-processing` verandert of wanneer de videorender stopt.


In [ ]:

def show_tail(path, title, length=12000):
    print(f"\n===== {title} =====\n")
    if not path.exists():
        print(f"Logbestand ontbreekt: {path}")
        return
    print(path.read_text(encoding="utf-8", errors="replace")[-length:])

print(f"API-proces actief: {globals().get('api_proc') is not None and api_proc.poll() is None}")
print(
    "Worker-proces actief: "
    f"{globals().get('github_worker_proc') is not None and github_worker_proc.poll() is None}"
)
print(f"API-poort: {globals().get('API_PORT', 'onbekend')}")
print(
    "Tunnel: "
    f"{getattr(globals().get('api_tunnel'), 'public_url', 'niet actief')}"
)

show_tail(API_LOG_PATH, "MoneyPrinterTurbo API")
show_tail(WORKER_LOG_PATH, "GitHub Issues worker")
